# Rank Classifier Training (MobileNetV3-Small)

Train a MobileNetV3-Small model to classify playing card **ranks** (13 classes: A, 2–10, J, Q, K).

This is Stage 2 of the 24 Game Solver ML pipeline:
- **Stage 1** (YOLO): Detect and crop individual cards from a camera frame
- **Stage 2** (this notebook): Classify the cropped card image into one of 13 ranks

The trained model is exported to **TFLite FP16** for on-device inference in the Flutter app.

> **Runtime**: GPU recommended. In Colab: Runtime → Change runtime type → T4 GPU.

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q tensorflow matplotlib seaborn kaggle scikit-learn pyyaml Pillow

import os
import shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
# Mount Google Drive to persist trained models across runtime disconnects
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/24_game_models"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Models will be saved to: {DRIVE_DIR}")

## 2. Download & Organize Dataset

We reuse the same [Playing Cards Object Detection Dataset](https://www.kaggle.com/datasets/andy8744/playing-cards-object-detection-dataset) from Notebook 01. The dataset has 52 per-card classes (e.g. `ace of clubs`, `king of hearts`). We:

1. Download the dataset (if not already present from Notebook 01)
2. Crop each card using its YOLO bounding box annotation
3. Organize crops into 13 rank-only folders (A, 2–10, J, Q, K)

**Before running:** Set your Kaggle API token:
```python
import os
os.environ["KAGGLE_API_TOKEN"] = "your_token_here"
```

In [ ]:
# Download the dataset (skip if already downloaded in Notebook 01)
DATA_DIR = "data"
if not os.path.isdir(os.path.join(DATA_DIR, "train")):
    !kaggle datasets download -d andy8744/playing-cards-object-detection-dataset -p {DATA_DIR} --unzip
    print("Dataset downloaded.")
else:
    print("Dataset already present, skipping download.")

import shutil, yaml

original_yaml = os.path.join(DATA_DIR, "data_original.yaml")
if not os.path.exists(original_yaml):
    src_yaml = os.path.join(DATA_DIR, "data.yaml")
    assert os.path.exists(src_yaml), "data.yaml not found — dataset download may have failed"
    shutil.copy(src_yaml, original_yaml)
    print("Created data_original.yaml from downloaded data.yaml")

    for split in ["train", "test", "valid"]:
        src = os.path.join(DATA_DIR, split, "labels")
        dst = os.path.join(DATA_DIR, split, "labels_original")
        if os.path.isdir(src) and not os.path.isdir(dst):
            shutil.copytree(src, dst)
            print(f"Backed up {split}/labels/ → {split}/labels_original/")

with open(original_yaml) as f:
    ds_config = yaml.safe_load(f)

# names can be a list ["10c", "Ac", ...] or a dict {0: "10c", ...}
raw_names = ds_config.get("names", [])
if isinstance(raw_names, list):
    class_names_52 = {i: name for i, name in enumerate(raw_names)}
else:
    class_names_52 = {int(k): v for k, v in raw_names.items()}

print(f"Found {len(class_names_52)} classes in dataset")
for i in list(class_names_52.items())[:5]:
    print(f"  {i}")

# Handles two naming conventions used by different versions of this dataset:
#   Full names:   "ace of clubs", "two of hearts", "jack of spades", ...
#   Abbreviated:  "Ac", "2h", "Jd", "Qs", "Ks", "10c", ...  (rank + suit letter)
FULL_NAME_MAP = {
    "ace": "A", "two": "2", "three": "3", "four": "4", "five": "5",
    "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
    "jack": "J", "queen": "Q", "king": "K",
}
ABBREV_RANK_MAP = {
    "a": "A", "2": "2", "3": "3", "4": "4", "5": "5",
    "6": "6", "7": "7", "8": "8", "9": "9", "10": "10",
    "j": "J", "q": "Q", "k": "K",
}

def class_name_to_rank(name: str) -> str:
    n = name.lower().strip()
    # Full-name format: "ace of clubs", "two of hearts", etc.
    for key, rank in FULL_NAME_MAP.items():
        if key in n:
            return rank
    # Abbreviated format: last char is suit (c/d/h/s), prefix is rank
    if len(n) >= 2 and n[-1] in "cdhs":
        prefix = n[:-1]
        if prefix in ABBREV_RANK_MAP:
            return ABBREV_RANK_MAP[prefix]
    return None

id_to_rank = {}
for cid, cname in class_names_52.items():
    rank = class_name_to_rank(str(cname))
    if rank:
        id_to_rank[int(cid)] = rank
    else:
        print(f"  WARNING: Could not map class {cid}: '{cname}'")

print(f"\nMapped {len(id_to_rank)} class IDs to ranks")
print(f"Ranks: {sorted(set(id_to_rank.values()))}")

### Crop Cards from Bounding Boxes

Use the YOLO annotations to crop individual cards from each image, then save them into rank-based folders. This creates a clean classification dataset from the detection dataset.

In [ ]:
# Crop cards from images using original YOLO bounding boxes and organize by rank.
# Reads from labels_original/ (backed up by Notebook 01 before class remapping)
# so class IDs 0–51 are intact and can be mapped to ranks.
from PIL import Image

RANK_DIR = "data/rank_cards"
os.makedirs(RANK_DIR, exist_ok=True)
for rank in ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]:
    os.makedirs(os.path.join(RANK_DIR, rank), exist_ok=True)

total_crops = 0

for split in ["train", "valid", "test"]:
    img_dir = os.path.join(DATA_DIR, split, "images")
    lbl_dir = os.path.join(DATA_DIR, split, "labels_original")  # original class IDs

    if not os.path.isdir(img_dir) or not os.path.isdir(lbl_dir):
        print(f"Skipping {split}: img_dir or labels_original/ not found")
        continue

    for lbl_file in sorted(os.listdir(lbl_dir)):
        if not lbl_file.endswith(".txt"):
            continue

        stem = Path(lbl_file).stem
        img_path = None
        for ext in [".jpg", ".jpeg", ".png"]:
            candidate = os.path.join(img_dir, stem + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break
        if img_path is None:
            continue

        img = Image.open(img_path)
        w, h = img.size

        with open(os.path.join(lbl_dir, lbl_file)) as f:
            for line_idx, line in enumerate(f):
                parts = line.strip().split()
                if len(parts) < 5:
                    continue

                class_id = int(parts[0])
                rank = id_to_rank.get(class_id)
                if rank is None:
                    continue

                cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = int((cx - bw / 2) * w)
                y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w)
                y2 = int((cy + bh / 2) * h)

                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(w, x2), min(h, y2)

                if x2 - x1 < 10 or y2 - y1 < 10:
                    continue

                crop = img.crop((x1, y1, x2, y2))
                out_path = os.path.join(RANK_DIR, rank, f"{split}_{stem}_{line_idx}.jpg")
                crop.save(out_path, quality=95)
                total_crops += 1

print(f"\nCropped {total_crops} card images into rank folders:")
for rank in sorted(os.listdir(RANK_DIR)):
    count = len(os.listdir(os.path.join(RANK_DIR, rank)))
    print(f"  {rank}: {count} images")

## 3. Synthetic Data Check

Before loading the dataset, verify that synthetic German card images exist.
If missing, run **02_german_card_synth.ipynb** first — it generates ~500 synthetic B (Jack) and D (Queen) crop images that improve classification of German-style cards.

In [ ]:
# Check for synthetic German card data (run 02_german_card_synth.ipynb first if missing)
from pathlib import Path

synth_j = list(Path("data/rank_cards/J").glob("synth_*.jpg")) if Path("data/rank_cards/J").exists() else []
synth_q = list(Path("data/rank_cards/Q").glob("synth_*.jpg")) if Path("data/rank_cards/Q").exists() else []

if not synth_j or not synth_q:
    print("WARNING: No synthetic German card data found.")
    print("Run 02_german_card_synth.ipynb first to generate synthetic J and Q images.")
    print("This improves classification of J and Q ranks on German-style cards.")
else:
    print(f"Synthetic data ready: {len(synth_j)} J images, {len(synth_q)} Q images.")

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32
RANK_DIR = "data/rank_cards"

# Class names in sorted order — this determines the index→rank mapping
CLASS_NAMES = sorted(os.listdir(RANK_DIR))
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")
assert len(CLASS_NAMES) == 13, f"Expected 13 classes, got {len(CLASS_NAMES)}"

# Load datasets with augmentation for training
train_ds = tf.keras.utils.image_dataset_from_directory(
    RANK_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    RANK_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
)

# Normalize to [0, 1]
normalization = tf.keras.layers.Rescaling(1.0 / 255)
train_ds = train_ds.map(lambda x, y: (normalization(x), y))
val_ds = val_ds.map(lambda x, y: (normalization(x), y))

# Data augmentation (applied only during training)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.04),          # ±15 degrees
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomZoom(0.1),
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))

# Prefetch for performance
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

print(f"Training batches: {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"Validation batches: {tf.data.experimental.cardinality(val_ds).numpy()}")

## 4. Build Model

MobileNetV3-Small pretrained on ImageNet, with a custom classification head for 13 rank classes. The backbone is frozen initially and then unfrozen for fine-tuning.

In [ ]:
# MobileNetV3-Small with custom classification head
base = tf.keras.applications.MobileNetV3Small(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base.trainable = False  # freeze backbone initially

model = tf.keras.Sequential([
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(13, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()
print(f"\nTotal params: {model.count_params():,}")

## 5. Resume from Checkpoint

Check Google Drive for existing checkpoints. If found, load weights and skip the corresponding training phase — useful after a Colab disconnect.

In [ ]:
import json

# Resume from Drive checkpoint if available
PHASE1_CKPT = os.path.join(DRIVE_DIR, "rank_classifier_phase1.keras")
PHASE2_CKPT = os.path.join(DRIVE_DIR, "rank_classifier_best.keras")
PHASE2_LAST_CKPT = os.path.join(DRIVE_DIR, "rank_classifier_last.keras")
PHASE2_STATE = os.path.join(DRIVE_DIR, "training_state.json")

SKIP_PHASE1 = False
RESUME_EPOCH = 0

if os.path.exists(PHASE2_LAST_CKPT):
    print(f"Found Phase 2 last checkpoint: {PHASE2_LAST_CKPT}")
    model = tf.keras.models.load_model(PHASE2_LAST_CKPT)
    SKIP_PHASE1 = True
    if os.path.exists(PHASE2_STATE):
        with open(PHASE2_STATE) as f:
            RESUME_EPOCH = json.load(f).get("phase2_epoch", 0)
    print(f"Resuming Phase 2 from epoch {RESUME_EPOCH}.")
elif os.path.exists(PHASE2_CKPT):
    # last.keras missing but best exists (e.g. first run saved best before disconnect)
    print(f"Found Phase 2 best checkpoint: {PHASE2_CKPT}")
    model = tf.keras.models.load_model(PHASE2_CKPT)
    SKIP_PHASE1 = True
    if os.path.exists(PHASE2_STATE):
        with open(PHASE2_STATE) as f:
            RESUME_EPOCH = json.load(f).get("phase2_epoch", 0)
    print(f"Resuming Phase 2 from epoch {RESUME_EPOCH} (from best checkpoint).")
elif os.path.exists(PHASE1_CKPT):
    print(f"Found Phase 1 checkpoint: {PHASE1_CKPT}")
    model.load_weights(PHASE1_CKPT)
    SKIP_PHASE1 = True
    print("Phase 1 skipped — head weights loaded. Proceeding to Phase 2 fine-tuning.")
else:
    print("No checkpoint found on Drive — starting fresh.")

## 6. Train Phase 1: Frozen Backbone

Train only the classification head for 5 epochs. The ImageNet backbone weights are frozen — this quickly teaches the head to map MobileNetV3 features to card ranks.

Checkpoint saved to Drive after each improvement so training can resume after a disconnect.

In [ ]:
# Phase 1: Train only the classification head (backbone frozen)
if SKIP_PHASE1:
    print("Skipping Phase 1 — checkpoint already loaded.")
else:
    history_frozen = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=5,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                PHASE1_CKPT,
                monitor="val_accuracy",
                save_best_only=True,
                verbose=1,
            ),
        ],
    )
    print(f"Phase 1 val accuracy: {history_frozen.history['val_accuracy'][-1]:.4f}")
    print(f"Phase 1 checkpoint saved to: {PHASE1_CKPT}")

## 7. Train Phase 2: Fine-tune

Unfreeze the full backbone and fine-tune at a lower learning rate (1e-4). Two checkpoints saved to Drive each epoch:
- **best**: saved only when `val_accuracy` improves
- **last**: always overwritten — use this to resume after a disconnect

In [ ]:
import json

# Phase 2: Unfreeze backbone and fine-tune at lower learning rate.
# Unfreeze via the loaded model's layers — base variable may be orphaned if model was
# replaced by load_model() in the resume cell.
for layer in model.layers:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

class SaveTrainingState(tf.keras.callbacks.Callback):
    """Writes completed epoch count to Drive so training can resume after a disconnect."""
    def on_epoch_end(self, epoch, logs=None):
        with open(PHASE2_STATE, "w") as f:
            json.dump({"phase2_epoch": epoch + 1}, f)

history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=RESUME_EPOCH,
    epochs=25,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=5, restore_best_weights=True
        ),
        tf.keras.callbacks.ModelCheckpoint(
            PHASE2_CKPT,
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.ModelCheckpoint(
            PHASE2_LAST_CKPT,
            save_best_only=False,
            verbose=0,
        ),
        SaveTrainingState(),
    ],
)

print(f"Phase 2 best val accuracy: {max(history_finetune.history['val_accuracy']):.4f}")
print(f"Best checkpoint: {PHASE2_CKPT}")
print(f"Last checkpoint: {PHASE2_LAST_CKPT}")

## 7. Evaluate: Confusion Matrix

Visualize per-class performance and flag systematic confusions (e.g., 6 vs 9, J/Q/K face cards). The confusion matrix is saved to `outputs/classifier/confusion_matrix.png`.

In [ ]:
# Confusion matrix to catch systematic errors (6/9, J/Q/K)
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
accuracy = np.mean(y_true == y_pred)
print(f"Overall accuracy: {accuracy:.4f}")
print(f"\nPer-class report:\n")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Rank Classifier Confusion Matrix (accuracy: {accuracy:.2%})")
plt.tight_layout()
os.makedirs("outputs/classifier", exist_ok=True)
plt.savefig("outputs/classifier/confusion_matrix.png", dpi=150)
plt.show()

# Flag common confusions
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        if i != j and cm[i][j] > 2:
            print(f"  WARNING: {CLASS_NAMES[i]} confused with {CLASS_NAMES[j]} ({cm[i][j]} times)")

## 8. Export to TFLite

Convert the trained Keras model to TFLite with FP16 quantization. FP16 halves the model size with negligible accuracy loss, making it suitable for on-device inference.

Output files:
- `outputs/classifier/rank_classifier.keras` — full Keras model (for resuming training)
- `outputs/classifier/rank_classifier.tflite` — quantized TFLite model (for Flutter app)
- `outputs/classifier/class_names.json` — index-to-rank mapping

In [ ]:
# Export to TFLite with FP16 quantization
os.makedirs("outputs/classifier", exist_ok=True)
model.save("outputs/classifier/rank_classifier.keras")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

tflite_path = "outputs/classifier/rank_classifier.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f"TFLite model saved: {tflite_path} ({size_mb:.1f} MB)")

# Save class name mapping for reference
import json
class_map = {i: name for i, name in enumerate(CLASS_NAMES)}
with open("outputs/classifier/class_names.json", "w") as f:
    json.dump(class_map, f, indent=2)
print(f"Class mapping: {class_map}")
print("\nCopy rank_classifier.tflite to flutter_app/assets/models/rank_classifier.tflite")

In [ ]:
# Save exports to Google Drive
import shutil

shutil.copy(tflite_path, os.path.join(DRIVE_DIR, "rank_classifier.tflite"))
shutil.copy("outputs/classifier/rank_classifier.keras", os.path.join(DRIVE_DIR, "rank_classifier.keras"))
shutil.copy("outputs/classifier/class_names.json", os.path.join(DRIVE_DIR, "class_names.json"))

print(f"All exports saved to {DRIVE_DIR}/")
for f in os.listdir(DRIVE_DIR):
    size = os.path.getsize(os.path.join(DRIVE_DIR, f)) / (1024 * 1024)
    print(f"  {f}: {size:.1f} MB")

## 9. Verify TFLite Model

Run a quick sanity check by loading the TFLite model and running inference on a few validation images. This confirms the exported model produces correct predictions before deploying to the Flutter app.

In [ ]:
# Quick sanity check: run TFLite model on a few validation images
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input:  shape={input_details[0]['shape']}, dtype={input_details[0]['dtype']}")
print(f"Output: shape={output_details[0]['shape']}, dtype={output_details[0]['dtype']}")

# Test on 5 validation images
correct = 0
total = 0
for images, labels in val_ds.take(1):
    for i in range(min(5, len(images))):
        img = images[i:i+1].numpy().astype(np.float32)
        interpreter.set_tensor(input_details[0]["index"], img)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])
        pred_class = np.argmax(output[0])
        true_class = labels[i].numpy()
        match = "✓" if pred_class == true_class else "✗"
        print(f"  {match} True: {CLASS_NAMES[true_class]}, Predicted: {CLASS_NAMES[pred_class]} "
              f"(conf: {output[0][pred_class]:.2f})")
        if pred_class == true_class:
            correct += 1
        total += 1

print(f"\nTFLite quick check: {correct}/{total} correct")

## 10. Fine-tune on Real Crops

Re-train the classifier on **real card corner crops** extracted by the YOLO pipeline from actual photos.

**Why:** The original model was trained on synthetic/Kaggle data. Real corner crops have different lighting, blur, and card styles (especially German-deck faces) that cause misclassifications.

**Data sources in `data/real_crops/`:**
- `2–6`: corner crops from Kaggle single-card photos (international deck, all 4 suits × 2 corners)
- `7–A, J, Q, K`: corner crops extracted via YOLO from real German-deck photos

**Two-phase strategy:**
1. **Freeze backbone** → train head on real crops only (fast convergence, no catastrophic forgetting)
2. **Unfreeze all** → fine-tune on mixed real + synthetic data at a low LR

**Before running:** upload `data/real_crops/` (zipped) to Drive as `24_game_models/real_crops.zip`, or mount Drive and copy the folder manually.

In [ ]:
import zipfile, json

# ── Paths ────────────────────────────────────────────────────────────────────
REAL_CROPS_ZIP   = os.path.join(DRIVE_DIR, "real_crops.zip")
REAL_CROPS_DIR   = "data/real_crops"
FT_BEST_CKPT     = os.path.join(DRIVE_DIR, "rank_classifier_ft_best.keras")
FT_LAST_CKPT     = os.path.join(DRIVE_DIR, "rank_classifier_ft_last.keras")
FT_STATE         = os.path.join(DRIVE_DIR, "training_state_ft.json")
FT_TFLITE_PATH   = "outputs/classifier/rank_classifier_ft.tflite"

# ── Unzip real crops from Drive (skip if already extracted) ──────────────────
if not os.path.isdir(REAL_CROPS_DIR):
    assert os.path.exists(REAL_CROPS_ZIP), (
        f"Not found: {REAL_CROPS_ZIP}\n"
        "Upload data/real_crops.zip to your Drive 24_game_models/ folder first."
    )
    with zipfile.ZipFile(REAL_CROPS_ZIP, "r") as z:
        z.extractall("data/")
    print(f"Extracted real crops to {REAL_CROPS_DIR}/")
else:
    print(f"Real crops already present at {REAL_CROPS_DIR}/")

# ── Summarise crops per rank ─────────────────────────────────────────────────
print("\nReal crops per rank:")
total_real = 0
for rank in sorted(os.listdir(REAL_CROPS_DIR)):
    n = len(os.listdir(os.path.join(REAL_CROPS_DIR, rank)))
    print(f"  {rank:>2}: {n}")
    total_real += n
print(f"  Total: {total_real}")

# ── Load existing model + resume epoch ───────────────────────────────────────
RESUME_FT_EPOCH = 0
if os.path.exists(FT_LAST_CKPT):
    ft_model_path = FT_LAST_CKPT
    if os.path.exists(FT_STATE):
        with open(FT_STATE) as f:
            RESUME_FT_EPOCH = json.load(f).get("ft2_epoch", 0)
    print(f"Resuming Phase FT-2 from epoch {RESUME_FT_EPOCH} ({FT_LAST_CKPT})")
elif os.path.exists(FT_BEST_CKPT):
    ft_model_path = FT_BEST_CKPT
    if os.path.exists(FT_STATE):
        with open(FT_STATE) as f:
            RESUME_FT_EPOCH = json.load(f).get("ft2_epoch", 0)
    print(f"Resuming Phase FT-2 from epoch {RESUME_FT_EPOCH} ({FT_BEST_CKPT})")
else:
    ft_model_path = PHASE2_CKPT
    print("No FT checkpoint found — starting fine-tuning from base model.")

assert os.path.exists(ft_model_path), (
    f"No trained model found at {ft_model_path}.\n"
    "Run sections 2–7 first to train the base model."
)
ft_model = tf.keras.models.load_model(ft_model_path)
print(f"Loaded model from: {ft_model_path}")

In [ ]:
# ── Build fine-tuning datasets ───────────────────────────────────────────────
#
# Class names MUST match the original training order (sorted alphabetical).
# Both real_crops/ and rank_cards/ use the same folder names, so
# image_dataset_from_directory assigns identical label indices.
#
# Validation strategy: we do NOT split real_crops — it has only ~104 samples
# (all from clean Kaggle photos the base model already knows), so a held-out
# split gives artificially perfect accuracy.  Instead we reuse val_ds from
# section 3 (20% of the full synthetic dataset) — larger, balanced, unseen
# during fine-tuning.  The true performance test is test_pipeline.py --batch
# on the 24-examples photos.

FT_IMG_SIZE   = 128
FT_BATCH_SIZE = 32
FT_SEED       = 99

# All real crops for training (no validation split)
real_train_ds = tf.keras.utils.image_dataset_from_directory(
    REAL_CROPS_DIR,
    seed=FT_SEED,
    image_size=(FT_IMG_SIZE, FT_IMG_SIZE),
    batch_size=FT_BATCH_SIZE,
    label_mode="int",
)

# Synthetic crops — used in mixed dataset for Phase FT-2
synth_ds = tf.keras.utils.image_dataset_from_directory(
    RANK_DIR,
    seed=FT_SEED,
    image_size=(FT_IMG_SIZE, FT_IMG_SIZE),
    batch_size=FT_BATCH_SIZE,
    label_mode="int",
)

# Verify class ordering is identical across all sources
assert real_train_ds.class_names == synth_ds.class_names == CLASS_NAMES, (
    f"Class name mismatch!\n  real:  {real_train_ds.class_names}\n"
    f"  synth: {synth_ds.class_names}\n"
    f"  original: {CLASS_NAMES}"
)
print(f"Class names: {real_train_ds.class_names}")

# Normalise all to [0, 1]
norm = tf.keras.layers.Rescaling(1.0 / 255)
real_train_ds = real_train_ds.map(lambda x, y: (norm(x), y))
synth_ds      = synth_ds.map(lambda x, y:      (norm(x), y))
# val_ds is already normalised from section 3

# Augmentation for training splits
augment = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.04),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomZoom(0.1),
])
real_train_ds = real_train_ds.map(lambda x, y: (augment(x, training=True), y))
synth_ds      = synth_ds.map(lambda x, y:      (augment(x, training=True), y))

# Mixed dataset: real crops weighted 60%, synthetic 40%
# Repeat both and take a fixed number of steps per epoch
mixed_ds = tf.data.Dataset.sample_from_datasets(
    [synth_ds.repeat(), real_train_ds.repeat()],
    weights=[0.4, 0.6],
    seed=FT_SEED,
).take(len(synth_ds) * 2)

real_train_ds = real_train_ds.prefetch(tf.data.AUTOTUNE)
mixed_ds      = mixed_ds.prefetch(tf.data.AUTOTUNE)
# val_ds already prefetched from section 3

print(f"\nReal train batches : {tf.data.experimental.cardinality(real_train_ds).numpy()}")
print(f"Mixed train batches: {tf.data.experimental.cardinality(mixed_ds).numpy()}")
print(f"Validation source  : val_ds from section 3 ({tf.data.experimental.cardinality(val_ds).numpy()} batches)")

In [ ]:
# ── Phase FT-1: Freeze backbone, train head on real crops only ───────────────
# Teaches the head to map real-world features to ranks without disturbing
# the backbone weights.

for layer in ft_model.layers:
    layer.trainable = False
# Unfreeze only the final Dense (classification head)
ft_model.layers[-1].trainable = True

ft_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

print("Phase FT-1: head-only training on real crops")
ft_model.summary()

history_ft1 = ft_model.fit(
    real_train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=4, restore_best_weights=True
        ),
    ],
)
print(f"Phase FT-1 best val accuracy: {max(history_ft1.history['val_accuracy']):.4f}")

In [ ]:
# ── Phase FT-2: Unfreeze all, fine-tune on mixed real + synthetic data ───────
# Low LR (1e-5) prevents catastrophic forgetting of synthetic knowledge while
# adapting backbone features to real card appearance.

for layer in ft_model.layers:
    layer.trainable = True

ft_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

class SaveFtState(tf.keras.callbacks.Callback):
    """Writes completed FT-2 epoch count to Drive for disconnect-safe resumption."""
    def on_epoch_end(self, epoch, logs=None):
        with open(FT_STATE, "w") as f:
            json.dump({"ft2_epoch": epoch + 1}, f)

print(f"Phase FT-2: full model fine-tuning on mixed data (real 60% + synthetic 40%)")
print(f"Starting from epoch {RESUME_FT_EPOCH}")

history_ft2 = ft_model.fit(
    mixed_ds,
    validation_data=val_ds,
    initial_epoch=RESUME_FT_EPOCH,
    epochs=20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=5, restore_best_weights=True
        ),
        tf.keras.callbacks.ModelCheckpoint(
            FT_BEST_CKPT, monitor="val_accuracy", save_best_only=True, verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            FT_LAST_CKPT, save_best_only=False, verbose=0
        ),
        SaveFtState(),
    ],
)
print(f"Phase FT-2 best val accuracy: {max(history_ft2.history['val_accuracy']):.4f}")

In [ ]:
# ── Export fine-tuned model to TFLite ───────────────────────────────────────
os.makedirs("outputs/classifier", exist_ok=True)

converter = tf.lite.TFLiteConverter.from_keras_model(ft_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_ft = converter.convert()

with open(FT_TFLITE_PATH, "wb") as f:
    f.write(tflite_ft)

size_mb = os.path.getsize(FT_TFLITE_PATH) / (1024 * 1024)
print(f"Fine-tuned TFLite saved: {FT_TFLITE_PATH} ({size_mb:.1f} MB)")

# Save to Drive
import shutil
shutil.copy(FT_TFLITE_PATH, os.path.join(DRIVE_DIR, "rank_classifier_ft.tflite"))
print(f"Copied to Drive: {DRIVE_DIR}/rank_classifier_ft.tflite")

# ── Confusion matrix on synthetic val set ────────────────────────────────────
# Note: val_ds (synthetic) measures whether fine-tuning hurt base performance.
# The real-world test is: run test_pipeline.py --batch on the 24-examples photos.
from sklearn.metrics import confusion_matrix, classification_report

y_true, y_pred = [], []
for images, labels in val_ds:
    preds = ft_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true, y_pred = np.array(y_true), np.array(y_pred)
accuracy = np.mean(y_true == y_pred)
print(f"\nFine-tuned val accuracy (synthetic): {accuracy:.4f}")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Fine-tuned Classifier — Synthetic Val Set (accuracy: {accuracy:.2%})")
plt.tight_layout()
plt.savefig("outputs/classifier/confusion_matrix_ft.png", dpi=150)
plt.show()

print("\nNext: copy rank_classifier_ft.tflite to flutter_app/assets/models/rank_classifier.tflite")
print("Then: run test_pipeline.py --batch data/24-examples/test.csv to measure real-world accuracy")